# Multi-Seed Zero-M AUC Analysis — 64-Neuron 4-Class

Runs the Zero-M (NuMIT) W/M pipeline for **3 seeds** [101, 202, 210], then:

1. **Per-draw AUC** of W-vs-k and M-vs-k curves via log-spaced trapezoidal integration
2. **Cohen's d at each k** between homo and hetero networks
3. **Identifies optimal subset sizes** for discriminating homogeneous from heterogeneous networks

The goal: determine which k (subset size) gives the best signal-to-noise ratio for detecting the effect of heterogeneity.

## 1. Imports & Configuration

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

import hashlib, io, itertools, json, math, time as time_module, warnings
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.linalg import solve_discrete_lyapunov
from scipy.optimize import root_scalar
from scipy.stats import norm, rankdata, wishart
from IPython.display import display

import importlib
import seeded_runs_common as seeded_runs_common
seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
build_seeded_pair = seeded_runs_common.build_seeded_pair
get_pair_checkpoint_paths = seeded_runs_common.get_pair_checkpoint_paths
load_default_caches = seeded_runs_common.load_default_caches

RUN_LABEL = "seeded_run_1_64n"
TASK_KEY = "4class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
OUTPUT_DIR = RUN_DIR / "zero_m_zscores"
OUTPUT_DIR.mkdir(exist_ok=True)

seeded_runs_common.BASE_PRMS["nb_recurrent"] = 64

SEEDS = [101, 202, 210]
SUBSET_SIZES = [2, 4, 8, 16, 25]
SUBSET_SAMPLE_SIZE = 50
N_NULL = 20
MAX_ATTEMPTS = 900
LAG = 1
RIDGE = 1e-2
WM_N_BATCHES = 8
WM_STRIDE = 4

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}")
print(f"Subset sizes: {SUBSET_SIZES}")
print(f"Output: {OUTPUT_DIR}")

## 2. W/M Estimation Engine

Same robust engine as the ZeroM Zscore notebook — Gaussian copula + multi-strategy fallback + Zero-M VAR(1) null.

In [ ]:
from zero_m_engine import (
    _stable_seed, compute_wm_from_spike_matrix,
    build_numit_null_distribution, score_observed_vs_null_m,
    sample_random_subsets,
)

def extract_hidden_traces(model, prms, data_cache, n_batches=8, layer_idx=0):
    mem_list, spk_list = [], []
    model.eval()
    with torch.no_grad():
        units, times, labels = data_cache.units, data_cache.times, data_cache.labels
        class_list = prms.get("class_list", list(range(20)))
        label_arr = labels if isinstance(labels, np.ndarray) else np.array(labels[:])
        sample_idx = np.where(np.isin(label_arr, class_list))[0]
        batch_size = min(prms.get("batch_size", 256), len(sample_idx))
        n_batches = min(n_batches, len(sample_idx) // max(batch_size, 1))
        for b in range(n_batches):
            bi = sample_idx[b * batch_size:(b + 1) * batch_size]
            actual_bs = len(bi)
            t_arrays = [np.round(times[idx] * (1.0 / prms["time_step"])).astype(np.int64) for idx in bi]
            u_arrays = [units[idx] for idx in bi]
            all_ts = np.concatenate(t_arrays); all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), [len(a) for a in t_arrays])
            valid = all_ts < prms["nb_steps"]
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            index_tensor = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            values = torch.ones(all_ts.size, dtype=torch.float32)
            x_batch = torch.sparse_coo_tensor(
                index_tensor, values,
                torch.Size([actual_bs, prms["nb_steps"], prms["nb_inputs"]])
            ).to_dense().clamp_(max=1.0).to(DEVICE)
            layer_recs = model(0, 0, x_batch)
            layer_out = layer_recs[layer_idx]
            spk, mem = layer_out[0], layer_out[1]
            mem_list.append(mem.permute(2, 0, 1).reshape(mem.shape[2], -1).cpu().numpy())
            spk_list.append(spk.permute(2, 0, 1).reshape(spk.shape[2], -1).cpu().numpy())
    return np.concatenate(mem_list, axis=1), np.concatenate(spk_list, axis=1)

print("✓ Engine imported. Ready.")

## 3. Multi-Seed Pipeline

Runs the full observed W/M sweep for each seed. Uses per-seed CSV checkpoints so interrupted runs can resume.

In [ ]:
def run_seed_observed(wm_seed):
    """Extract traces, filter active, run observed W/M sweep for one seed."""
    obs_csv = OUTPUT_DIR / f"observed_wm_seed{wm_seed}.csv"
    if obs_csv.exists():
        print(f"  Seed {wm_seed}: loading cached {obs_csv.name}")
        return pd.read_csv(obs_csv)
    
    print(f"\n{'='*60}")
    print(f"  Seed {wm_seed}")
    print(f"{'='*60}")
    
    train_cache, test_cache = load_default_caches()
    wm_pair = build_seeded_pair(master_seed=wm_seed, task_key=TASK_KEY, mem_distribution_family=MEM_DISTRIBUTION_FAMILY)
    paths = get_pair_checkpoint_paths(master_seed=wm_seed, run_label=RUN_LABEL, task_key=TASK_KEY,
                                       mem_distribution_family=MEM_DISTRIBUTION_FAMILY, checkpoint_root=CHECKPOINT_ROOT)
    
    homo_ckpt = torch.load(paths["homogeneous_anchor"], map_location="cpu")
    hetero_ckpt = torch.load(paths["heterogeneous_sampled"], map_location="cpu")
    wm_pair["hom_model"].load_state_dict(homo_ckpt["model_state_dict"])
    wm_pair["hetero_model"].load_state_dict(hetero_ckpt["model_state_dict"])
    
    print(f"  Homo acc={homo_ckpt['history']['test_acc'][-1]:.3f}, Hetero acc={hetero_ckpt['history']['test_acc'][-1]:.3f}")
    
    homo_mem, homo_spk = extract_hidden_traces(wm_pair["hom_model"], wm_pair["hom_prms"], test_cache, n_batches=WM_N_BATCHES)
    hetero_mem, hetero_spk = extract_hidden_traces(wm_pair["hetero_model"], wm_pair["hetero_prms"], test_cache, n_batches=WM_N_BATCHES)
    homo_spk, hetero_spk = homo_spk[:, ::WM_STRIDE], hetero_spk[:, ::WM_STRIDE]
    
    homo_active = np.where(homo_spk.mean(axis=1) >= 1e-9)[0]
    hetero_active = np.where(hetero_spk.mean(axis=1) >= 1e-9)[0]
    print(f"  Active: homo={len(homo_active)}, hetero={len(hetero_active)}")
    
    active_spk = {"homo": homo_spk[homo_active,:].astype(np.float64),
                  "hetero": hetero_spk[hetero_active,:].astype(np.float64)}
    
    observed_rows = []
    for net_key, spk in [("homo", active_spk["homo"]), ("hetero", active_spk["hetero"])]:
        nb = int(spk.shape[0])
        sizes = [k for k in SUBSET_SIZES if k <= nb]
        for k in sizes:
            subs, total = sample_random_subsets(nb, k, SUBSET_SAMPLE_SIZE, _stable_seed(net_key, "subset", k))
            for draw, idx in enumerate(subs, 1):
                try:
                    w, m, _ = compute_wm_from_spike_matrix(spk[list(idx),:], lag=LAG, ridge=RIDGE)
                    st = "ok"
                except Exception as e:
                    w, m, st = np.nan, np.nan, f"err:{e}"
                observed_rows.append({"network":net_key, "subset_size":k, "subset_draw":draw,
                    "seed":wm_seed, "observed_W_bits":w if np.isfinite(w) else np.nan,
                    "observed_M_bits":m if np.isfinite(m) else np.nan, "status":st})
    
    df = pd.DataFrame(observed_rows)
    df.to_csv(obs_csv, index=False)
    print(f"  Saved {len(df)} rows to {obs_csv.name}")
    return df

# ── Run all seeds ──────────────────────────────────────────────────
all_obs_dfs = []
for seed in SEEDS:
    df = run_seed_observed(seed)
    all_obs_dfs.append(df)

all_obs_df = pd.concat(all_obs_dfs, ignore_index=True)
print(f"\n✓ All {len(SEEDS)} seeds complete. Total rows: {len(all_obs_df)}")
display(all_obs_df.groupby(["seed","network","subset_size"]).agg(
    W_mean=("observed_W_bits","mean"), M_mean=("observed_M_bits","mean"),
    ok=("status", lambda x: (x=="ok").sum())).round(4))

## 4. Per-Draw AUC (Log-Spaced Trapezoidal Integration)

For each subset draw, we compute the area under the W-vs-k (and M-vs-k) curve using log₂(k) spacing. This AUC captures the total information capacity of that neuron subset.

In [ ]:
def compute_auc(k_vals, y_vals):
    """Log₂-spaced trapezoidal AUC."""
    x = np.log2(np.asarray(k_vals, dtype=np.float64))
    return float(np.trapz(np.asarray(y_vals, dtype=np.float64), x))

# ── Per-draw AUC ──────────────────────────────────────────────────
auc_rows = []
for (seed, network, draw), grp in all_obs_df.groupby(["seed", "network", "subset_draw"]):
    grp = grp.sort_values("subset_size")
    kw = grp["observed_W_bits"].values
    km = grp["observed_M_bits"].values
    ks = grp["subset_size"].values
    auc_rows.append({
        "seed": seed, "network": network, "subset_draw": draw,
        "AUC_W": compute_auc(ks, kw),
        "AUC_M": compute_auc(ks, km),
    })

auc_df = pd.DataFrame(auc_rows)
print("AUC distributions (log₂-k trapezoidal):")
display(auc_df.groupby(["seed", "network"]).agg(
    AUC_W_mean=("AUC_W","mean"), AUC_W_std=("AUC_W","std"),
    AUC_M_mean=("AUC_M","mean"), AUC_M_std=("AUC_M","std")).round(4))

## 5. Cohen's d at Each k + Plots

Computes effect size between homo/hetero W and M at each subset size, then visualises AUC distributions and discrimination power.

In [ ]:
# ── Cohen's d at each k ───────────────────────────────────────────
from scipy import stats

cohens_rows = []
for k in sorted(all_obs_df["subset_size"].unique()):
    sub = all_obs_df[all_obs_df["subset_size"] == k]
    h_w = sub[sub["network"]=="homo"]["observed_W_bits"].dropna()
    t_w = sub[sub["network"]=="hetero"]["observed_W_bits"].dropna()
    h_m = sub[sub["network"]=="homo"]["observed_M_bits"].dropna()
    t_m = sub[sub["network"]=="hetero"]["observed_M_bits"].dropna()
    
    d_w = (t_w.mean() - h_w.mean()) / np.sqrt((t_w.var() + h_w.var()) / 2) if len(h_w)>1 and len(t_w)>1 else np.nan
    d_m = (t_m.mean() - h_m.mean()) / np.sqrt((t_m.var() + h_m.var()) / 2) if len(h_m)>1 and len(t_m)>1 else np.nan
    
    # Per-seed Cohen's d
    for seed in SEEDS:
        ss = sub[sub["seed"]==seed]
        shw = ss[ss["network"]=="homo"]["observed_W_bits"].dropna()
        stw = ss[ss["network"]=="hetero"]["observed_W_bits"].dropna()
        shm = ss[ss["network"]=="homo"]["observed_M_bits"].dropna()
        stm = ss[ss["network"]=="hetero"]["observed_M_bits"].dropna()
        sd_w = (stw.mean()-shw.mean())/np.sqrt((stw.var()+shw.var())/2) if len(shw)>1 and len(stw)>1 else np.nan
        sd_m = (stm.mean()-shm.mean())/np.sqrt((stm.var()+shm.var())/2) if len(shm)>1 and len(stm)>1 else np.nan
        cohens_rows.append({"k":k, "seed":seed, "Cohens_d_W":sd_w, "Cohens_d_M":sd_m})
    
    cohens_rows.append({"k":k, "seed":"pooled", "Cohens_d_W":d_w, "Cohens_d_M":d_m})

cohens_df = pd.DataFrame(cohens_rows)
pooled = cohens_df[cohens_df["seed"]=="pooled"].set_index("k")

print("Cohen's d (pooled across seeds):")
display(pooled[["Cohens_d_W","Cohens_d_M"]].round(4))

# ── Best k for discrimination ─────────────────────────────────────
best_w = pooled["Cohens_d_W"].abs().idxmax()
best_m = pooled["Cohens_d_M"].abs().idxmax()
print(f"\nBest k for W-based discrimination: k={best_w} (d={pooled.loc[best_w,'Cohens_d_W']:.3f})")
print(f"Best k for M-based discrimination: k={best_m} (d={pooled.loc[best_m,'Cohens_d_M']:.3f})")

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# (0,0) W vs k — all seeds
ax = axes[0,0]
for seed, marker in zip(SEEDS, ['o','s','^']):
    for net, color, ls in [("homo","#2196F3","-"),("hetero","#FF5722","--")]:
        sub = all_obs_df[(all_obs_df["seed"]==seed)&(all_obs_df["network"]==net)]
        grp = sub.groupby("subset_size")["observed_W_bits"]
        ks = grp.mean().index.values
        ax.errorbar(ks, grp.mean(), yerr=grp.sem(), color=color, marker=marker,
                    ls=ls, ms=6, lw=1.5, capsize=3, alpha=0.7,
                    label=f"{net}_s{seed}" if seed==SEEDS[0] else "")
ax.set_xlabel("k"); ax.set_ylabel("W (bits)")
ax.set_title("W-Information vs k — All Seeds", fontweight="bold")
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# (0,1) M vs k
ax = axes[0,1]
for seed, marker in zip(SEEDS, ['o','s','^']):
    for net, color, ls in [("homo","#2196F3","-"),("hetero","#FF5722","--")]:
        sub = all_obs_df[(all_obs_df["seed"]==seed)&(all_obs_df["network"]==net)]
        grp = sub.groupby("subset_size")["observed_M_bits"]
        ks = grp.mean().index.values
        ax.errorbar(ks, grp.mean(), yerr=grp.sem(), color=color, marker=marker,
                    ls=ls, ms=6, lw=1.5, capsize=3, alpha=0.7)
ax.set_xlabel("k"); ax.set_ylabel("M (bits)")
ax.set_title("M-Information vs k", fontweight="bold")
ax.grid(True, alpha=0.3)

# (0,2) AUC_W boxplots
ax = axes[0,2]
positions = []
data_list, colors_list, labels_list = [], [], []
for i, seed in enumerate(SEEDS):
    for j, (net, color) in enumerate([("homo","#2196F3"),("hetero","#FF5722")]):
        pos = i*3 + j*0.8
        positions.append(pos)
        vals = auc_df[(auc_df["seed"]==seed)&(auc_df["network"]==net)]["AUC_W"].dropna().values
        data_list.append(vals)
        colors_list.append(color)
        labels_list.append(f"{net}_s{seed}")
bp = ax.boxplot(data_list, positions=positions, widths=0.5, patch_artist=True, showfliers=True)
for patch, color in zip(bp["boxes"], colors_list):
    patch.set_facecolor(color); patch.set_alpha(0.5)
ax.set_xticks([i*3+0.4 for i in range(len(SEEDS))])
ax.set_xticklabels([f"Seed {s}" for s in SEEDS])
ax.set_ylabel("AUC_W (log₂-k trapz)")
ax.set_title("AUC W-Information Distributions", fontweight="bold")
ax.legend([plt.Rectangle((0,0),1,1,fc="#2196F3",alpha=0.5),
           plt.Rectangle((0,0),1,1,fc="#FF5722",alpha=0.5)],
          ["Homo","Hetero"], fontsize=7)
ax.grid(True, alpha=0.3, axis="y")

# (1,0) Cohen's d vs k
ax = axes[1,0]
ks_plot = sorted(pooled.index)
dw_bar = ax.bar(np.arange(len(ks_plot))-0.15, [pooled.loc[k,"Cohens_d_W"] for k in ks_plot],
                0.3, color="#2196F3", alpha=0.7, label="Cohen's d W")
dm_bar = ax.bar(np.arange(len(ks_plot))+0.15, [pooled.loc[k,"Cohens_d_M"] for k in ks_plot],
                0.3, color="#FF5722", alpha=0.7, label="Cohen's d M")
# Overlay per-seed points
for seed, marker in zip(SEEDS, ['o','s','^']):
    sub = cohens_df[cohens_df["seed"]==seed]
    for j, k in enumerate(ks_plot):
        row = sub[sub["k"]==k]
        if len(row)>0:
            ax.plot(j-0.15, row["Cohens_d_W"].values[0], marker=marker, color="black", ms=6)
            ax.plot(j+0.15, row["Cohens_d_M"].values[0], marker=marker, color="black", ms=6)
ax.axhline(y=0, color="black", lw=0.5)
ax.set_xticks(range(len(ks_plot))); ax.set_xticklabels([str(k) for k in ks_plot])
ax.set_xlabel("Subset size k"); ax.set_ylabel("Cohen's d")
ax.set_title("Effect Size (Hetero − Homo) vs k", fontweight="bold")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# (1,1) ΔW (hetero−homo) per draw
ax = axes[1,1]
delta_by_k = []
for k in ks_plot:
    sub = all_obs_df[all_obs_df["subset_size"]==k]
    h_w = sub[sub["network"]=="homo"].groupby(["seed","subset_draw"])["observed_W_bits"].mean()
    t_w = sub[sub["network"]=="hetero"].groupby(["seed","subset_draw"])["observed_W_bits"].mean()
    common = h_w.index.intersection(t_w.index)
    dw = t_w[common].values - h_w[common].values
    delta_by_k.append(dw)
ax.boxplot(delta_by_k, tick_labels=[str(k) for k in ks_plot], showfliers=True)
ax.axhline(y=0, color="black", lw=0.5)
ax.set_xlabel("Subset size k"); ax.set_ylabel("ΔW (hetero − homo) bits")
ax.set_title("W Difference Distribution per k", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

# (1,2) Summary table as text
ax = axes[1,2]
ax.axis("off")
summary_text = "Best k for Discrimination:\n\n"
summary_text += f"  W-based:  k={best_w}  d={pooled.loc[best_w,'Cohens_d_W']:.3f}\n"
summary_text += f"  M-based:  k={best_m}  d={pooled.loc[best_m,'Cohens_d_M']:.3f}\n\n"
summary_text += "Cohen's d (↑ = hetero higher):\n"
for k in ks_plot:
    dw = pooled.loc[k,"Cohens_d_W"]
    dm = pooled.loc[k,"Cohens_d_M"]
    star_w = " ★" if k==best_w else ""
    star_m = " ★" if k==best_m else ""
    summary_text += f"  k={k:2d}:  d_W={dw:+.3f}{star_w}  d_M={dm:+.3f}{star_m}\n"
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontfamily="monospace",
        fontsize=9, verticalalignment="top", bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plot_path = OUTPUT_DIR / "multiseed_auc_analysis.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {plot_path}")

## 6. Interpretation

**How to read the results:**

- **AUC_W / AUC_M**: The area under the W-vs-k (or M-vs-k) curve via log₂-spaced trapezoidal integration. Higher AUC means more total information across all subset sizes. Compare homo vs hetero AUC distributions to see if heterogeneity systematically increases information capacity.

- **Cohen's d at each k**: Measures the standardized difference between homo and hetero at a given k. Positive d means hetero > homo. The k with the **largest absolute Cohen's d** is the best subset size for discriminating between network types.

- **ΔW distribution per k**: The raw difference (hetero − homo) in W-information. A distribution shifted above zero at a given k means hetero has reliably higher W at that subset size.

**Recommended action:**
- Focus comparisons on the k with the highest |Cohen's d| — this gives the best statistical power to detect heterogeneity effects.
- If both W and M point to the same k, that's the most robust choice.
- Use AUC distributions to confirm that heterogeneity effects are consistent across seeds.